In [ ]:
!pip install -q wfdb
import wfdb, os

for db in ["mitdb", "nstdb", "incartdb"]:
    wfdb.dl_database(db, dl_dir=f"/kaggle/working/data/{db}")

mit = sorted({f[:3] for f in os.listdir("/kaggle/working/data/mitdb") if f.endswith(".dat")})
print(len(mit), "MIT-BIH records")   # expect 48

rec = wfdb.rdrecord("/kaggle/working/data/mitdb/100")
ann = wfdb.rdann("/kaggle/working/data/mitdb/100", "atr")
print(rec.fs, rec.sig_name, len(ann.symbol), "annotations")   # 360 ['MLII', 'V5'] ~2274

In [ ]:
import numpy as np, pandas as pd, wfdb, os
from math import gcd
from scipy.signal import butter, sosfiltfilt, resample_poly

DATA = "/kaggle/working/data"
FS = 250                                   # must match the Arduino firmware
PRE, POST = int(0.36 * FS), int(0.64 * FS) # 90 + 160 = 250-sample beat window
MARGIN = 8                                 # ±32 ms jitter room for augmentation
CLASSES = ["N", "S", "V", "F"]

AAMI = {**dict.fromkeys("NLRej", "N"), **dict.fromkeys("AaJS", "S"),
        **dict.fromkeys("VE", "V"), "F": "F", **dict.fromkeys("/fQ", "Q")}
BEAT_SYMBOLS = list(AAMI)
PACED = {"102", "104", "107", "217"}

DS1 = ["101","106","108","109","112","114","115","116","118","119","122",
       "124","201","203","205","207","208","209","215","220","223","230"]
DS2 = ["100","103","105","111","113","117","121","123","200","202","210",
       "212","213","214","219","221","222","228","231","232","233","234"]


def preprocess(x, fs_in, fs_out=FS):
    x = np.asarray(x, dtype=np.float64)
    if fs_in != fs_out:
        g = gcd(int(fs_in), int(fs_out))
        x = resample_poly(x, fs_out // g, int(fs_in) // g)
    sos = butter(3, [0.5, 40], btype="band", fs=fs_out, output="sos")
    return sosfiltfilt(sos, x)


def causal_mean(v, n):
    c = np.cumsum(np.r_[0.0, v])
    i = np.arange(len(v))
    s = np.maximum(0, i - n + 1)
    return (c[i + 1] - c[s]) / (i + 1 - s)


def rr_features(r, fs=FS):
    rr = np.diff(r) / fs
    pre = np.r_[rr[0], rr]
    post = np.r_[rr, rr[-1]]
    loc10, loc100 = causal_mean(pre, 10), causal_mean(pre, 100)
    f = np.stack([pre, post, loc10, pre / loc10, post / loc10, pre / post, pre / loc100], 1)
    return np.log(np.clip(f, 1e-2, 10)).astype(np.float32)


def beats_from_record(sig, fs_in, samples, symbols, rec_id):
    x = preprocess(sig, fs_in)
    r = np.round(np.asarray(samples) * FS / fs_in).astype(int)
    sym = np.asarray(symbols)
    is_beat = np.isin(sym, BEAT_SYMBOLS)
    r, sym = r[is_beat], sym[is_beat]
    rr = rr_features(r)
    lab = np.array([AAMI[s] for s in sym])
    keep = (np.isin(lab, CLASSES)
            & (r - PRE - MARGIN >= 0) & (r + POST + MARGIN <= len(x)))
    idx = r[keep, None] + np.arange(-PRE - MARGIN, POST + MARGIN)[None, :]
    return dict(W=x[idx].astype(np.float32), RR=rr[keep],
                y=np.array([CLASSES.index(l) for l in lab[keep]], dtype=np.int64),
                rec=np.full(keep.sum(), rec_id), r=r[keep])


def load(db, rec, lead):
    p = f"{DATA}/{db}/{rec}"
    s, a = wfdb.rdrecord(p), wfdb.rdann(p, "atr")
    return beats_from_record(s.p_signal[:, s.sig_name.index(lead)], s.fs, a.sample, a.symbol, rec)


def build(db, records, lead):
    parts = [load(db, r, lead) for r in records]
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}


mit = build("mitdb", DS1 + DS2, "MLII")
incart = build("incartdb", [f"I{i:02d}" for i in range(1, 76)], "II")

os.makedirs("/kaggle/working/cache", exist_ok=True)
np.savez_compressed("/kaggle/working/cache/mitdb_beats.npz", **mit)
np.savez_compressed("/kaggle/working/cache/incart_beats.npz", **incart)

def counts(d, mask=None):
    y = d["y"] if mask is None else d["y"][mask]
    return {c: int((y == i).sum()) for i, c in enumerate(CLASSES)}

print("window shape:", mit["W"].shape, "| RR features:", mit["RR"].shape)
print("DS1   :", counts(mit, np.isin(mit["rec"], DS1)))
print("DS2   :", counts(mit, np.isin(mit["rec"], DS2)))
print("INCART:", counts(incart))

In [ ]:
import matplotlib.pyplot as plt
t = (np.arange(mit["W"].shape[1]) - PRE - MARGIN) / FS
plt.figure(figsize=(10, 4))
for i, c in enumerate(CLASSES):
    m = mit["W"][mit["y"] == i]
    plt.plot(t, m.mean(0), label=f"{c} (n={len(m)})")
plt.axvline(0, ls="--", c="gray"); plt.xlabel("seconds from R-peak"); plt.legend(); plt.show()

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, random, time, json
import lightgbm as lgb
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
L = PRE + POST  # 250
print("device:", DEV)

def load_cache(path):
    d = dict(np.load(path))
    d["Wt"] = torch.tensor(d["W"], device=DEV)
    d["RRt"] = torch.tensor(d["RR"], device=DEV)
    d["yt"] = torch.tensor(d["y"], device=DEV)
    return d

mit = load_cache("/kaggle/working/cache/mitdb_beats.npz")
incart = load_cache("/kaggle/working/cache/incart_beats.npz")

bank = []
for name in ["em", "ma", "bw"]:
    s = wfdb.rdrecord(f"{DATA}/nstdb/{name}")
    for ch in range(s.p_signal.shape[1]):
        n = preprocess(s.p_signal[:, ch], s.fs)
        bank.append(n / (n.std() + 1e-9))
NOISE = torch.tensor(np.concatenate(bank), dtype=torch.float32, device=DEV)
AR = torch.arange(L, device=DEV)


def make_batch(W, j, train):
    xb = W[j]
    B = xb.shape[0]
    off = (torch.randint(0, 2 * MARGIN + 1, (B,), device=DEV) if train
           else torch.full((B,), MARGIN, device=DEV))
    x = xb.gather(1, off[:, None] + AR[None])
    if train:
        sig_std = x.std(1, keepdim=True) + 1e-6
        start = torch.randint(0, NOISE.numel() - L, (B,), device=DEV)
        noise = NOISE[start[:, None] + AR[None]]
        snr_db = torch.empty(B, 1, device=DEV).uniform_(6, 30)
        apply = (torch.rand(B, 1, device=DEV) < 0.7).float()
        x = x + apply * noise * sig_std / (10 ** (snr_db / 20))
    x = (x - x.median(1, keepdim=True).values) / (x.std(1, keepdim=True) + 1e-6)
    return x[:, None, :]

print("noise bank:", NOISE.numel(), "samples | MIT on GPU:", tuple(mit["Wt"].shape))

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride, k=7, p=0.1):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, k, stride, k // 2, bias=False); self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, k, 1, k // 2, bias=False); self.b2 = nn.BatchNorm1d(cout)
        self.drop = nn.Dropout(p)
        self.skip = (nn.Identity() if cin == cout and stride == 1 else
                     nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout)))

    def forward(self, x):
        h = self.drop(F.relu(self.b1(self.c1(x))))
        return F.relu(self.b2(self.c2(h)) + self.skip(x))


class Head(nn.Module):
    def __init__(self, emb, use_rr, n_rr=7, n_cls=4):
        super().__init__()
        self.use_rr = use_rr
        self.rr = nn.Sequential(nn.Linear(n_rr, 32), nn.ReLU()) if use_rr else nn.Identity()
        self.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(emb + (32 if use_rr else 0), 64),
                                nn.ReLU(), nn.Linear(64, n_cls))

    def forward(self, e, rr):
        if self.use_rr:
            e = torch.cat([e, self.rr(rr)], 1)
        return self.fc(e)


class ResNet1D(nn.Module):
    def __init__(self, use_rr=True, w=32):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, w, 15, 1, 7, bias=False), nn.BatchNorm1d(w), nn.ReLU())
        self.body = nn.Sequential(ResBlock(w, w, 1), ResBlock(w, 2 * w, 2), ResBlock(2 * w, 2 * w, 1),
                                  ResBlock(2 * w, 4 * w, 2), ResBlock(4 * w, 4 * w, 1), ResBlock(4 * w, 4 * w, 2))
        self.head = Head(8 * w, use_rr)

    def forward(self, x, rr):
        h = self.body(self.stem(x))
        return self.head(torch.cat([h.mean(-1), h.amax(-1)], 1), rr)


class CNNBiLSTM(nn.Module):
    def __init__(self, use_rr=True):
        super().__init__()
        def blk(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                                 nn.BatchNorm1d(o), nn.ReLU(), nn.MaxPool1d(2))
        self.cnn = nn.Sequential(blk(1, 32, 7), blk(32, 64, 5), blk(64, 64, 5))
        self.lstm = nn.LSTM(64, 64, batch_first=True, bidirectional=True)
        self.att = nn.Linear(128, 1)
        self.head = Head(128, use_rr)

    def forward(self, x, rr):
        h, _ = self.lstm(self.cnn(x).transpose(1, 2))
        a = torch.softmax(self.att(h), dim=1)
        return self.head((a * h).sum(1), rr)


CANDIDATES = {
    "resnet_rr":     lambda: ResNet1D(use_rr=True),
    "resnet_noRR":   lambda: ResNet1D(use_rr=False),   # ablation: proves RR features matter
    "cnn_bilstm_rr": lambda: CNNBiLSTM(use_rr=True),
    "lgbm":          None,
}
for k, mk in CANDIDATES.items():
    if mk:
        print(k, sum(p.numel() for p in mk().parameters()), "params")

In [ ]:
CFG = dict(epochs=40, batch=512, lr=2e-3, wd=1e-3, patience=7,
           cw_power=0.5, label_smoothing=0.05, lgbm_trees=2000)


def class_weights(y, power):
    c = np.bincount(y, minlength=4).astype(float)
    w = np.where(c > 0, (c.sum() / (4 * np.maximum(c, 1))) ** power, 0.0)
    return w / w[c > 0].mean()


def macro_f1(y, p):
    return f1_score(y, p, labels=np.unique(y), average="macro", zero_division=0)


@torch.no_grad()
def predict_proba(model, D, idx, bs=4096):
    model.eval()
    out = []
    for i in range(0, len(idx), bs):
        j = torch.as_tensor(idx[i:i + bs], device=DEV)
        with torch.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
            logits = model(make_batch(D["Wt"], j, False), D["RRt"][j])
        out.append(torch.softmax(logits.float(), 1).cpu().numpy())
    return np.concatenate(out)


def train_net(make_model, D, tr_idx, va_idx, cfg=CFG, verbose=False):
    model = make_model().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    steps = int(np.ceil(len(tr_idx) / cfg["batch"]))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"], total_steps=cfg["epochs"] * steps)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    cw = torch.tensor(class_weights(D["y"][tr_idx], cfg["cw_power"]), dtype=torch.float32, device=DEV)
    tr = torch.as_tensor(tr_idx, device=DEV)
    best, best_state, bad = -1.0, None, 0
    for ep in range(cfg["epochs"]):
        model.train()
        t0 = time.time()
        perm = tr[torch.randperm(len(tr), device=DEV)]
        for i in range(0, len(perm), cfg["batch"]):
            j = perm[i:i + cfg["batch"]]
            with torch.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
                logits = model(make_batch(D["Wt"], j, True), D["RRt"][j])
                loss = F.cross_entropy(logits.float(), D["yt"][j], weight=cw,
                                       label_smoothing=cfg["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        f1 = macro_f1(D["y"][va_idx], predict_proba(model, D, va_idx).argmax(1))
        if f1 > best:
            best, bad = f1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose:
            print(f"  ep {ep + 1:02d} loss {loss.item():.4f} val macroF1 {f1:.4f} ({time.time() - t0:.1f}s)")
        if bad >= cfg["patience"]:
            break
    model.load_state_dict(best_state)
    return model, best, ep + 1


def handcrafted(D, idx):
    W = D["W"][idx][:, MARGIN:MARGIN + L]
    W = (W - np.median(W, 1, keepdims=True)) / (W.std(1, keepdims=True) + 1e-6)
    q = W[:, PRE - 15:PRE + 15]                                  # ±60 ms around R
    return np.hstack([
        W.reshape(len(W), 50, 5).mean(2),                        # 50-point downsampled beat
        D["RR"][idx],
        W.max(1, keepdims=True), W.min(1, keepdims=True),
        (np.abs(q) > 0.5 * np.abs(q).max(1, keepdims=True)).sum(1, keepdims=True),  # QRS width proxy
        (q ** 2).sum(1, keepdims=True),                          # QRS energy
        (W[:, PRE + 25:] ** 2).mean(1, keepdims=True),           # T-wave energy
    ]).astype(np.float32)


def train_lgbm(D, tr_idx, va_idx, cfg=CFG):
    ytr = D["y"][tr_idx]
    cw = class_weights(ytr, cfg["cw_power"])
    m = lgb.LGBMClassifier(objective="multiclass", n_estimators=cfg["lgbm_trees"], learning_rate=0.05,
                           num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           random_state=SEED, verbose=-1)
    m.fit(handcrafted(D, tr_idx), ytr, sample_weight=cw[ytr],
          eval_set=[(handcrafted(D, va_idx), D["y"][va_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m


def fit_predict(name, D, tr, va, te, cfg=CFG, verbose=False):
    if name == "lgbm":
        m = train_lgbm(D, tr, va, cfg)
        P = np.zeros((len(te), 4))
        P[:, m.classes_] = m.predict_proba(handcrafted(D, te))
        return m, P
    m, _, _ = train_net(CANDIDATES[name], D, tr, va, cfg, verbose)
    return m, predict_proba(m, D, te)


def report(y, p, name=""):
    cm = confusion_matrix(y, p, labels=range(4))
    tp = np.diag(cm)
    se = tp / np.maximum(cm.sum(1), 1)
    ppv = tp / np.maximum(cm.sum(0), 1)
    f1 = 2 * se * ppv / np.maximum(se + ppv, 1e-9)
    row = {"model": name, "acc": tp.sum() / cm.sum(), "macroF1": f1.mean()}
    for i, c in enumerate(CLASSES):
        row[f"{c}_Se"], row[f"{c}_+P"], row[f"{c}_F1"] = se[i], ppv[i], f1[i]
    return row, cm


def patient_split(idx, D, n_splits, seed):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(idx[a], idx[b]) for a, b in sgkf.split(idx, D["y"][idx], D["rec"][idx])]

In [ ]:
ds1_idx = np.where(np.isin(mit["rec"], DS1))[0]
tr, te = patient_split(ds1_idx, mit, 5, SEED)[0]
tr_in, va = patient_split(tr, mit, 4, SEED)[0]
print("test patients :", sorted(set(mit["rec"][te])))
print("val patients  :", sorted(set(mit["rec"][va])))
print("sizes train/val/test:", len(tr_in), len(va), len(te))

quick = {**CFG, "epochs": 3, "lgbm_trees": 200}
rows = []
for name in CANDIDATES:
    t0 = time.time()
    _, P = fit_predict(name, mit, tr_in, va, te, quick, verbose=True)
    row, cm = report(mit["y"][te], P.argmax(1), name)
    rows.append(row)
    print(f"{name}: {time.time() - t0:.0f}s")
pd.DataFrame(rows).round(3)

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*valid feature names.*")
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

K = 5
folds = patient_split(ds1_idx, mit, K, SEED)
cv = {name: {"y": [], "P": [], "fold": []} for name in CANDIDATES}
fold_rows = []

for f, (tr, te) in enumerate(folds):
    tr_in, va = patient_split(tr, mit, 5, SEED + f)[0]
    print(f"\nfold {f} | test {sorted(map(str, set(mit['rec'][te])))} "
          f"| test counts {np.bincount(mit['y'][te], minlength=4).tolist()}")
    for name in CANDIDATES:
        seed_all(SEED + f)
        t0 = time.time()
        _, P = fit_predict(name, mit, tr_in, va, te)
        y = mit["y"][te]
        cv[name]["y"].append(y); cv[name]["P"].append(P); cv[name]["fold"].append(np.full(len(y), f))
        row, _ = report(y, P.argmax(1), name)
        row["fold"] = f
        row["macroF1_present"] = macro_f1(y, P.argmax(1))
        fold_rows.append(row)
        print(f"   {name:14s} macroF1 {row['macroF1_present']:.3f} | "
              f"S_Se {row['S_Se']:.2f} V_Se {row['V_Se']:.2f} ({time.time() - t0:.0f}s)")

np.savez_compressed("/kaggle/working/cache/cv_predictions.npz",
                    **{f"{n}_{k}": np.concatenate(v[k]) for n, v in cv.items() for k in v})

In [ ]:
fold_df = pd.DataFrame(fold_rows)
summary, cms = [], {}
for name in CANDIDATES:
    y = np.concatenate(cv[name]["y"])
    P = np.concatenate(cv[name]["P"])
    row, cms[name] = report(y, P.argmax(1), name)
    s = fold_df.loc[fold_df.model == name, "macroF1_present"]
    row["fold_mean"], row["fold_std"] = s.mean(), s.std()
    summary.append(row)

summary = pd.DataFrame(summary).set_index("model").sort_values("macroF1", ascending=False)
BEST = summary.index[0]
print("Selected for DS2 evaluation:", BEST)

os.makedirs("/kaggle/working/export", exist_ok=True)
summary.round(4).to_csv("/kaggle/working/export/cv_summary.csv")
json.dump({"selected_model": BEST, "selection_rule": "highest pooled 5-fold patient-wise macro-F1 on DS1",
           "cfg": CFG}, open("/kaggle/working/export/selection.json", "w"), indent=2)
summary.round(3)

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, len(cms), figsize=(5 * len(cms), 4.2))
for ax, (name, cm) in zip(axes, cms.items()):
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    sns.heatmap(pct, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=100, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name}\nmacro-F1 {summary.loc[name, 'macroF1']:.3f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("/kaggle/working/export/cv_confusion.png", dpi=150); plt.show()

In [ ]:
baseline_summary = summary.copy(); baseline_summary.round(3)

In [ ]:
from tqdm.auto import tqdm

def zscore(W):
    return (W - np.median(W, 1, keepdims=True)) / (W.std(1, keepdims=True) + 1e-6)

def add_template(D, n_prev=20, desc=""):
    Z = zscore(D["W"])
    T = np.empty_like(Z)
    for rec in tqdm(pd.unique(D["rec"]), desc=f"templates {desc}"):
        idx = np.where(D["rec"] == rec)[0]
        Zr = Z[idx]
        T[idx[0]] = Zr[0]
        for k in range(1, len(idx)):
            T[idx[k]] = np.median(Zr[max(0, k - n_prev):k], 0)
    D["T"] = T
    D["Tt"] = torch.tensor(T, device=DEV)

add_template(mit, desc="MIT")
add_template(incart, desc="INCART")

In [ ]:
def _norm(x):
    return (x - x.median(1, keepdim=True).values) / (x.std(1, keepdim=True) + 1e-6)


def make_batch(D, j, train, use_tpl=False):
    xb = D["Wt"][j]
    B = xb.shape[0]
    off = (torch.randint(0, 2 * MARGIN + 1, (B,), device=DEV) if train
           else torch.full((B,), MARGIN, device=DEV))
    gi = off[:, None] + AR[None]
    x = xb.gather(1, gi)
    if train:
        sig_std = x.std(1, keepdim=True) + 1e-6
        start = torch.randint(0, NOISE.numel() - L, (B,), device=DEV)
        noise = NOISE[start[:, None] + AR[None]]
        snr_db = torch.empty(B, 1, device=DEV).uniform_(6, 30)
        apply = (torch.rand(B, 1, device=DEV) < 0.7).float()
        x = x + apply * noise * sig_std / (10 ** (snr_db / 20))
    x = _norm(x)
    if not use_tpl:
        return x[:, None, :]
    t = _norm(D["Tt"][j].gather(1, gi))
    return torch.stack([x, x - t], 1)


class ResNet1D(nn.Module):
    def __init__(self, use_rr=True, in_ch=1, w=32):
        super().__init__()
        self.in_ch = in_ch
        self.stem = nn.Sequential(nn.Conv1d(in_ch, w, 15, 1, 7, bias=False), nn.BatchNorm1d(w), nn.ReLU())
        self.body = nn.Sequential(ResBlock(w, w, 1), ResBlock(w, 2 * w, 2), ResBlock(2 * w, 2 * w, 1),
                                  ResBlock(2 * w, 4 * w, 2), ResBlock(4 * w, 4 * w, 1), ResBlock(4 * w, 4 * w, 2))
        self.head = Head(8 * w, use_rr)

    def forward(self, x, rr):
        h = self.body(self.stem(x))
        return self.head(torch.cat([h.mean(-1), h.amax(-1)], 1), rr)


class CNNBiLSTM(nn.Module):
    def __init__(self, use_rr=True, in_ch=1):
        super().__init__()
        self.in_ch = in_ch
        def blk(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                                 nn.BatchNorm1d(o), nn.ReLU(), nn.MaxPool1d(2))
        self.cnn = nn.Sequential(blk(in_ch, 32, 7), blk(32, 64, 5), blk(64, 64, 5))
        self.lstm = nn.LSTM(64, 64, batch_first=True, bidirectional=True)
        self.att = nn.Linear(128, 1)
        self.head = Head(128, use_rr)

    def forward(self, x, rr):
        h, _ = self.lstm(self.cnn(x).transpose(1, 2))
        a = torch.softmax(self.att(h), dim=1)
        return self.head((a * h).sum(1), rr)


CFG.update(epochs=35, lr=1e-3, patience=10, min_epochs=12)


@torch.no_grad()
def predict_proba(model, D, idx, bs=4096):
    model.eval()
    tpl = model.in_ch == 2
    out = []
    for i in range(0, len(idx), bs):
        j = torch.as_tensor(idx[i:i + bs], device=DEV)
        with torch.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
            logits = model(make_batch(D, j, False, tpl), D["RRt"][j])
        out.append(torch.softmax(logits.float(), 1).cpu().numpy())
    return np.concatenate(out)


def train_net(make_model, D, tr_idx, va_idx, cfg=CFG, desc="train"):
    model = make_model().to(DEV)
    tpl = model.in_ch == 2
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    steps = int(np.ceil(len(tr_idx) / cfg["batch"]))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=steps),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, (cfg["epochs"] - 1) * steps),
                                                   eta_min=cfg["lr"] * 0.02)], milestones=[steps])
    scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    cw = torch.tensor(class_weights(D["y"][tr_idx], cfg["cw_power"]), dtype=torch.float32, device=DEV)
    tr = torch.as_tensor(tr_idx, device=DEV)
    best, best_state, bad = -1.0, None, 0
    bar = tqdm(range(cfg["epochs"]), desc=desc, leave=False)
    for ep in bar:
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEV)]
        for i in range(0, len(perm), cfg["batch"]):
            j = perm[i:i + cfg["batch"]]
            with torch.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
                logits = model(make_batch(D, j, True, tpl), D["RRt"][j])
                loss = F.cross_entropy(logits.float(), D["yt"][j], weight=cw,
                                       label_smoothing=cfg["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        f1 = macro_f1(D["y"][va_idx], predict_proba(model, D, va_idx).argmax(1))
        if f1 > best:
            best, bad = f1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        bar.set_postfix(loss=f"{loss.item():.3f}", valF1=f"{f1:.3f}", best=f"{best:.3f}", wait=bad)
        if ep + 1 >= cfg["min_epochs"] and bad >= cfg["patience"]:
            break
    bar.close()
    model.load_state_dict(best_state)
    return model, best, ep + 1


def handcrafted(D, idx):
    W = zscore(D["W"][idx][:, MARGIN:MARGIN + L])
    T = zscore(D["T"][idx][:, MARGIN:MARGIN + L])
    n = len(W)
    d = W - T
    q, tq = W[:, PRE - 15:PRE + 15], T[:, PRE - 15:PRE + 15]
    return np.hstack([
        W.reshape(n, 50, 5).mean(2),
        d.reshape(n, 50, 5).mean(2),                               # difference from patient template
        D["RR"][idx],
        W.max(1, keepdims=True), W.min(1, keepdims=True),
        (np.abs(q) > 0.5 * np.abs(q).max(1, keepdims=True)).sum(1, keepdims=True),
        (q ** 2).sum(1, keepdims=True),
        (W[:, PRE + 25:] ** 2).mean(1, keepdims=True),
        (W * T).mean(1, keepdims=True),                            # similarity to template
        (q * tq).mean(1, keepdims=True),                           # QRS similarity to template
        np.sqrt((d ** 2).mean(1, keepdims=True)),
    ]).astype(np.float32)


def fit_predict(name, D, tr, va, te, cfg=CFG):
    if name.startswith("lgbm"):
        m = train_lgbm(D, tr, va, cfg)
        P = np.zeros((len(te), 4))
        P[:, m.classes_] = m.predict_proba(handcrafted(D, te))
        return m, P
    m, _, _ = train_net(CANDIDATES[name], D, tr, va, cfg, desc=name)
    return m, predict_proba(m, D, te)


CANDIDATES = {
    "resnet_rr":         lambda: ResNet1D(use_rr=True, in_ch=1),   # same model, fixed schedule
    "resnet_rr_tpl":     lambda: ResNet1D(use_rr=True, in_ch=2),
    "cnn_bilstm_rr_tpl": lambda: CNNBiLSTM(use_rr=True, in_ch=2),
    "lgbm_tpl":          None,
}

_, P = fit_predict("resnet_rr_tpl", mit, tr_in[:3000], va[:1000], te[:500],
                   {**CFG, "epochs": 2, "min_epochs": 1})
print("sanity OK:", P.shape)

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*valid feature names.*")
warnings.filterwarnings("ignore", message=".*lr_scheduler.step.*")

cv2 = {name: {"y": [], "P": [], "fold": []} for name in CANDIDATES}
fold_rows2 = []
outer = tqdm(total=K * len(CANDIDATES), desc="CV run 2 (fold x model)")
for f, (tr, te) in enumerate(folds):
    tr_in, va = patient_split(tr, mit, 5, SEED + f)[0]
    tqdm.write(f"\nfold {f} | test {sorted(map(str, set(mit['rec'][te])))}")
    for name in CANDIDATES:
        seed_all(SEED + f)
        t0 = time.time()
        outer.set_postfix(fold=f, model=name)
        _, P = fit_predict(name, mit, tr_in, va, te)
        y = mit["y"][te]
        cv2[name]["y"].append(y); cv2[name]["P"].append(P); cv2[name]["fold"].append(np.full(len(y), f))
        row, _ = report(y, P.argmax(1), name)
        row["fold"], row["macroF1_present"] = f, macro_f1(y, P.argmax(1))
        fold_rows2.append(row)
        tqdm.write(f"   {name:18s} macroF1 {row['macroF1_present']:.3f} | S_Se {row['S_Se']:.2f} "
                   f"V_Se {row['V_Se']:.2f} N_Se {row['N_Se']:.2f} ({time.time() - t0:.0f}s)")
        outer.update(1)
outer.close()

np.savez_compressed("/kaggle/working/cache/cv2_predictions.npz",
                    **{f"{n}_{k}": np.concatenate(v[k]) for n, v in cv2.items() for k in v})

In [ ]:
fold_df2 = pd.DataFrame(fold_rows2)
rows, cms2 = [], {}
for name in CANDIDATES:
    y, P = np.concatenate(cv2[name]["y"]), np.concatenate(cv2[name]["P"])
    row, cms2[name] = report(y, P.argmax(1), name)
    s = fold_df2.loc[fold_df2.model == name, "macroF1_present"]
    row["fold_mean"], row["fold_std"] = s.mean(), s.std()
    rows.append(row)
summary2 = pd.DataFrame(rows).set_index("model").sort_values("macroF1", ascending=False)

comparison = pd.concat([baseline_summary.assign(run="1_baseline"),
                        summary2.assign(run="2_template")]).reset_index()
cols = ["run", "model", "acc", "macroF1", "fold_mean", "fold_std",
        "N_Se", "N_+P", "S_Se", "S_+P", "V_Se", "V_+P", "F_Se", "F_+P"]
display(comparison[cols].sort_values("macroF1", ascending=False).round(3))

BEST = summary2.index[0]
print("Selected for DS2 evaluation:", BEST)
summary2.round(4).to_csv("/kaggle/working/export/cv2_summary.csv")
json.dump({"selected_model": BEST, "selection_rule": "highest pooled 5-fold patient-wise macro-F1 on DS1 (run 2)",
           "cfg": CFG}, open("/kaggle/working/export/selection.json", "w"), indent=2)

fig, axes = plt.subplots(1, len(cms2), figsize=(5 * len(cms2), 4.2))
for ax, (name, cm) in zip(axes, cms2.items()):
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    sns.heatmap(pct, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=100, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name}\nmacro-F1 {summary2.loc[name, 'macroF1']:.3f}")
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("/kaggle/working/export/cv2_confusion.png", dpi=150); plt.show()

In [ ]:
from torch.optim.swa_utils import AveragedModel

@torch.no_grad()
def recompute_bn(model, D, tr, tpl, n_batches=60, bs=1024):
    bns = [m for m in model.modules() if isinstance(m, nn.BatchNorm1d)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None
    model.train()
    perm = tr[torch.randperm(len(tr), device=DEV)]
    for i in range(0, min(len(perm), n_batches * bs), bs):
        j = perm[i:i + bs]
        model(make_batch(D, j, False, tpl), D["RRt"][j])
    model.eval()


def train_fixed(make_model, D, tr_idx, cfg, desc="train"):
    model = make_model().to(DEV)
    tpl = model.in_ch == 2
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    steps = int(np.ceil(len(tr_idx) / cfg["batch"]))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=steps),
        torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, (cfg["epochs"] - 1) * steps),
                                                   eta_min=cfg["lr"] * 0.01)], milestones=[steps])
    scaler = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    cw = torch.tensor(class_weights(D["y"][tr_idx], cfg["cw_power"]), dtype=torch.float32, device=DEV)
    tr = torch.as_tensor(tr_idx, device=DEV)
    swa = AveragedModel(model)
    swa_start = cfg["epochs"] - cfg["swa_epochs"]
    bar = tqdm(range(cfg["epochs"]), desc=desc, leave=False)
    for ep in bar:
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEV)]
        for i in range(0, len(perm), cfg["batch"]):
            j = perm[i:i + cfg["batch"]]
            with torch.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
                logits = model(make_batch(D, j, True, tpl), D["RRt"][j])
                loss = F.cross_entropy(logits.float(), D["yt"][j], weight=cw,
                                       label_smoothing=cfg["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        if ep >= swa_start:
            swa.update_parameters(model)
        bar.set_postfix(loss=f"{loss.item():.3f}", lr=f"{sched.get_last_lr()[0]:.1e}",
                        swa=ep >= swa_start)
    bar.close()
    final = swa.module
    recompute_bn(final, D, tr, tpl)
    return final


CFG3 = {**CFG, "epochs": 15, "swa_epochs": 5, "lr": 1e-3}

def fit_predict3(name, D, tr, te, cfg, seed):
    if name.startswith("lgbm"):
        tr_in, va = patient_split(tr, D, 5, seed)[0]
        m = train_lgbm(D, tr_in, va, cfg)
        P = np.zeros((len(te), 4))
        P[:, m.classes_] = m.predict_proba(handcrafted(D, te))
        return m, P
    m = train_fixed(CANDIDATES[name], D, tr, cfg, desc=name)
    return m, predict_proba(m, D, te)

In [ ]:
RUN3 = {
    "resnet_tpl_cw0.5": ("resnet_rr_tpl", {"cw_power": 0.5}),
    "resnet_tpl_cw1.0": ("resnet_rr_tpl", {"cw_power": 1.0}),
    "lgbm_tpl_cw1.0":   ("lgbm_tpl",      {"cw_power": 1.0}),
}

cv3 = {k: {"y": [], "P": [], "fold": []} for k in RUN3}
outer = tqdm(total=K * len(RUN3), desc="CV run 3 (fold x config)")
for f, (tr, te) in enumerate(folds):
    tqdm.write(f"\nfold {f} | test {sorted(map(str, set(mit['rec'][te])))}")
    for key, (name, over) in RUN3.items():
        seed_all(SEED + f)
        t0 = time.time()
        outer.set_postfix(fold=f, config=key)
        _, P = fit_predict3(name, mit, tr, te, {**CFG3, **over}, SEED + f)
        y = mit["y"][te]
        cv3[key]["y"].append(y); cv3[key]["P"].append(P); cv3[key]["fold"].append(np.full(len(y), f))
        row, _ = report(y, P.argmax(1), key)
        tqdm.write(f"   {key:18s} macroF1 {macro_f1(y, P.argmax(1)):.3f} | S_Se {row['S_Se']:.2f} "
                   f"V_Se {row['V_Se']:.2f} N_Se {row['N_Se']:.2f} F_Se {row['F_Se']:.2f} ({time.time() - t0:.0f}s)")
        outer.update(1)
outer.close()

np.savez_compressed("/kaggle/working/cache/cv3_predictions.npz",
                    **{f"{n}_{k}": np.concatenate(v[k]) for n, v in cv3.items() for k in v})

In [ ]:
def tune_class_bias(y, P, grid=np.linspace(-2, 3, 26), iters=3):
    logP = np.log(P + 1e-9)
    b = np.zeros(4)
    for _ in range(iters):
        for c in [1, 2, 3]:
            scores = []
            for g in grid:
                bb = b.copy(); bb[c] = g
                scores.append(report(y, (logP + bb).argmax(1))[0]["macroF1"])
            b[c] = grid[int(np.argmax(scores))]
    return b


def nested_bias_predict(y, P, fold):
    pred = np.empty(len(y), dtype=int)
    for f in tqdm(np.unique(fold), desc="nested bias", leave=False):
        m = fold != f
        b = tune_class_bias(y[m], P[m])
        pred[~m] = (np.log(P[~m] + 1e-9) + b).argmax(1)
    return pred


rows, cms3, biases = [], {}, {}
for key in tqdm(RUN3, desc="evaluating configs"):
    y, P, fold = (np.concatenate(cv3[key][k]) for k in ["y", "P", "fold"])
    row, cms3[f"{key} raw"] = report(y, P.argmax(1), f"{key} | raw")
    rows.append(row)
    row, cms3[f"{key} +bias"] = report(y, nested_bias_predict(y, P, fold), f"{key} | +bias (nested)")
    rows.append(row)
    biases[key] = tune_class_bias(y, P)

prev = summary2.loc[["resnet_rr_tpl"]].reset_index().assign(model="resnet_rr_tpl | run 2 (early stop)")
summary3 = pd.concat([pd.DataFrame(rows), prev], ignore_index=True).set_index("model")
cols = ["acc", "macroF1", "N_Se", "N_+P", "S_Se", "S_+P", "V_Se", "V_+P", "F_Se", "F_+P"]
summary3 = summary3[cols].sort_values("macroF1", ascending=False)
display(summary3.round(3))

best_row = summary3.index[0]
best_key = best_row.split(" | ")[0]
use_bias = "+bias" in best_row
FINAL = {
    "config": best_key,
    "model": RUN3[best_key][0] if best_key in RUN3 else "resnet_rr_tpl",
    "cfg": {**CFG3, **RUN3[best_key][1]} if best_key in RUN3 else CFG,
    "class_bias": biases[best_key].tolist() if use_bias and best_key in RUN3 else [0, 0, 0, 0],
    "selection_rule": "highest pooled 5-fold patient-wise macro-F1 on DS1 (bias tuned nested)",
}
print(json.dumps(FINAL, indent=2))
summary3.round(4).to_csv("/kaggle/working/export/cv3_summary.csv")
json.dump(FINAL, open("/kaggle/working/export/selection.json", "w"), indent=2)

show = [k for k in cms3 if k.startswith(best_key)]
fig, axes = plt.subplots(1, len(show), figsize=(5 * len(show), 4.2))
for ax, k in zip(np.atleast_1d(axes), show):
    cm = cms3[k]
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    sns.heatmap(pct, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=100, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(k); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("/kaggle/working/export/cv3_confusion.png", dpi=150); plt.show()

In [ ]:
FINAL = json.load(open("/kaggle/working/export/selection.json"))
CFG_FINAL, MODEL_NAME = FINAL["cfg"], FINAL["model"]
ds2_idx = np.where(np.isin(mit["rec"], DS2))[0]
inc_idx = np.arange(len(incart["y"]))
SEEDS = [42, 43, 44]

eval_models, rows, probs = [], [], {"DS2": [], "INCART": []}
for s in tqdm(SEEDS, desc="final DS1 models"):
    seed_all(s)
    m = train_fixed(CANDIDATES[MODEL_NAME], mit, ds1_idx, CFG_FINAL, desc=f"seed {s}")
    eval_models.append(m)
    torch.save(m.state_dict(), f"/kaggle/working/export/eval_ds1_seed{s}.pth")
    for name, D, idx in [("DS2", mit, ds2_idx), ("INCART", incart, inc_idx)]:
        P = predict_proba(m, D, idx)
        probs[name].append(P)
        row, _ = report(D["y"][idx], P.argmax(1), f"{name} seed{s}")
        row.update(set=name, seed=s)
        rows.append(row)
        tqdm.write(f"seed {s} {name:6s} macroF1 {row['macroF1']:.3f} | N_Se {row['N_Se']:.2f} "
                   f"S_Se {row['S_Se']:.2f} V_Se {row['V_Se']:.2f} F_Se {row['F_Se']:.2f}")

test_df = pd.DataFrame(rows)
metric_cols = ["acc", "macroF1", "N_Se", "N_+P", "S_Se", "S_+P", "V_Se", "V_+P", "F_Se", "F_+P"]
agg = test_df.groupby("set")[metric_cols].agg(["mean", "std"]).round(3)
display(agg)

In [ ]:
ens_rows, ens_cms = [], {}
for name, D, idx in tqdm([("DS2", mit, ds2_idx), ("INCART", incart, inc_idx)], desc="ensemble"):
    P = np.mean(probs[name], 0)
    row, ens_cms[name] = report(D["y"][idx], P.argmax(1), f"{name} | 3-seed ensemble")
    ens_rows.append(row)
display(pd.DataFrame(ens_rows).set_index("model")[metric_cols].round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (name, cm) in zip(axes, ens_cms.items()):
    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    sns.heatmap(pct, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=100, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(f"{name} (unseen patients) - ensemble"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("/kaggle/working/export/test_confusion.png", dpi=150); plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

all_idx = np.arange(len(mit["y"]))
tr_b, te_b = train_test_split(all_idx, test_size=0.3, stratify=mit["y"], random_state=SEED)
for _ in tqdm(range(1), desc="beat-wise split model"):
    seed_all(SEED)
    m_b = train_fixed(CANDIDATES[MODEL_NAME], mit, tr_b, CFG_FINAL, desc="beat-wise")
row_b, cm_b = report(mit["y"][te_b], predict_proba(m_b, mit, te_b).argmax(1), "beat-wise 70/30 (intra-patient)")

headline = pd.DataFrame([
    {**agg.loc["DS2"].xs("mean", level=1).to_dict(), "model": "DS2 inter-patient (mean of 3 seeds)"},
    {**agg.loc["INCART"].xs("mean", level=1).to_dict(), "model": "INCART external (mean of 3 seeds)"},
    row_b,
]).set_index("model")[metric_cols]
display(headline.round(4))

json.dump({"final_config": FINAL,
           "test_per_seed": test_df.to_dict(orient="records"),
           "test_ensemble": ens_rows,
           "beatwise_split": row_b},
          open("/kaggle/working/export/metrics_test.json", "w"), indent=2, default=float)

In [ ]:
P = np.mean(probs["DS2"], 0)
pred, y, recs = P.argmax(1), mit["y"][ds2_idx], mit["rec"][ds2_idx]
rows = []
for r in tqdm(DS2, desc="per-record S"):
    m = recs == r
    s = y[m] == 1
    if s.sum() == 0:
        continue
    rows.append(dict(rec=r, beats=m.sum(), S_beats=s.sum(),
                     S_as_S=(pred[m][s] == 1).sum(), S_as_N=(pred[m][s] == 0).sum(),
                     S_as_V=(pred[m][s] == 2).sum(), S_share_of_record=s.mean()))
diag = pd.DataFrame(rows).sort_values("S_beats", ascending=False)
diag["S_Se"] = diag.S_as_S / diag.S_beats
display(diag.round(3))

In [ ]:
from scipy.signal import find_peaks

def detect_r_peaks(x, fs=FS):
    sos = butter(2, [5, 18], btype="band", fs=fs, output="sos")
    y = sosfiltfilt(sos, x)
    e = np.gradient(y) ** 2
    w = max(1, int(round(0.12 * fs)))
    integ = np.convolve(e, np.ones(w) / w, mode="same")
    blk = int(5 * fs)
    thr = np.empty_like(integ)
    for s in range(0, len(integ), blk):
        thr[s:s + blk] = 0.3 * np.percentile(integ[max(0, s - 2 * blk):s + 3 * blk], 98)
    cand, _ = find_peaks(integ, distance=int(0.25 * fs))
    cand = cand[integ[cand] > thr[cand]]
    half = int(0.08 * fs)
    r = [max(0, p - half) + int(np.argmax(np.abs(x[max(0, p - half):p + half + 1]))) for p in cand]
    keep = []
    for i in sorted(set(r)):
        if not keep or i - keep[-1] > int(0.2 * fs):
            keep.append(i)
        elif abs(x[i]) > abs(x[keep[-1]]):
            keep[-1] = i
    return np.array(keep, dtype=int)


def match_peaks(det, ref, tol):
    j = np.searchsorted(ref, det)
    used = np.zeros(len(ref), bool)
    pairs = []
    for i, d in enumerate(det):
        best, bd = -1, tol + 1
        for k in (j[i] - 1, j[i]):
            if 0 <= k < len(ref) and not used[k] and abs(ref[k] - d) <= tol and abs(ref[k] - d) < bd:
                best, bd = k, abs(ref[k] - d)
        if best >= 0:
            used[best] = True
            pairs.append((i, best))
    return np.array(pairs, dtype=int).reshape(-1, 2)


def running_template(Z, n_prev=20):
    T = np.empty_like(Z)
    T[0] = Z[0]
    for k in range(1, len(Z)):
        T[k] = np.median(Z[max(0, k - n_prev):k], 0)
    return T


NSTDB_RAW = {n: wfdb.rdrecord(f"{DATA}/nstdb/{n}").p_signal[:, 0] for n in ["em", "ma"]}

def add_real_noise(raw, fs_in, snr_db, rng):
    ref_power = np.var(preprocess(raw, fs_in, fs_in))
    noise = np.zeros_like(raw)
    for n in NSTDB_RAW.values():
        st = rng.integers(0, len(n) - len(raw)) if len(n) > len(raw) else 0
        seg = np.resize(n[st:st + len(raw)], len(raw))
        noise += (seg - seg.mean()) / (seg.std() + 1e-9)
    noise /= noise.std() + 1e-9
    return raw + noise * np.sqrt(ref_power / 10 ** (snr_db / 10))


def end_to_end_record(rec, models, device_fs=None, snr_db=None, seed=0):
    s, a = wfdb.rdrecord(f"{DATA}/mitdb/{rec}"), wfdb.rdann(f"{DATA}/mitdb/{rec}", "atr")
    raw, fs_in = s.p_signal[:, s.sig_name.index("MLII")], s.fs
    if snr_db is not None:
        raw = add_real_noise(raw, fs_in, snr_db, np.random.default_rng(seed))
    if device_fs:
        raw = resample_poly(raw, device_fs, fs_in)
        fs_in = device_fs
    x = preprocess(raw, fs_in)

    sym = np.asarray(a.symbol)
    beat = np.isin(sym, BEAT_SYMBOLS)
    ref = np.round(a.sample[beat] * FS / s.fs).astype(int)
    ref_lab = np.array([AAMI[c] for c in sym[beat]])

    det = detect_r_peaks(x)
    pairs = match_peaks(det, ref, int(0.15 * FS))
    det_stats = dict(TP=len(pairs), FN=len(ref) - len(pairs), FP=len(det) - len(pairs))

    rr = rr_features(det)
    ok = (det - PRE - MARGIN >= 0) & (det + POST + MARGIN <= len(x))
    W = x[det[ok, None] + np.arange(-PRE - MARGIN, POST + MARGIN)[None, :]].astype(np.float32)
    D = dict(W=W, RR=rr[ok], T=running_template(zscore(W)))
    D.update(Wt=torch.tensor(D["W"], device=DEV), RRt=torch.tensor(D["RR"], device=DEV),
             Tt=torch.tensor(D["T"], device=DEV))
    P = np.mean([predict_proba(m, D, np.arange(len(W))) for m in models], 0)

    pos_in_ok = -np.ones(len(det), int)
    pos_in_ok[ok] = np.arange(ok.sum())
    y_true, y_pred = [], []
    for di, ri in pairs:
        if pos_in_ok[di] >= 0 and ref_lab[ri] in CLASSES:
            y_true.append(CLASSES.index(ref_lab[ri]))
            y_pred.append(int(P[pos_in_ok[di]].argmax()))
    return det_stats, np.array(y_true), np.array(y_pred)


st, yt, yp = end_to_end_record("100", eval_models)
print("record 100:", st, "| beats classified:", len(yt), "| acc:", (yt == yp).mean().round(3))

In [ ]:
CONDITIONS = [
    ("annotated R-peaks (reference)", None, None),
    ("end-to-end 250 Hz clean",       None, None),
    ("end-to-end 125 Hz device",      125,  None),
    ("end-to-end 60 Hz (prototype)",  60,   None),
    ("end-to-end 250 Hz, SNR 18 dB",  None, 18),
    ("end-to-end 250 Hz, SNR 12 dB",  None, 12),
    ("end-to-end 250 Hz, SNR 6 dB",   None, 6),
]

stress_rows = []
ref_row = [r for r in ens_rows if r["model"].startswith("DS2")][0]
stress_rows.append({**ref_row, "model": CONDITIONS[0][0], "det_Se": 1.0, "det_+P": 1.0})

for name, dev_fs, snr in tqdm(CONDITIONS[1:], desc="conditions"):
    TP = FN = FP = 0
    Y, Yp = [], []
    for i, rec in enumerate(tqdm(DS2, desc=name, leave=False)):
        stt, yt, yp = end_to_end_record(rec, eval_models, dev_fs, snr, seed=i)
        TP += stt["TP"]; FN += stt["FN"]; FP += stt["FP"]
        Y.append(yt); Yp.append(yp)
    row, _ = report(np.concatenate(Y), np.concatenate(Yp), name)
    row["det_Se"], row["det_+P"] = TP / (TP + FN), TP / (TP + FP)
    stress_rows.append(row)
    tqdm.write(f"{name:32s} det Se {row['det_Se']:.4f} +P {row['det_+P']:.4f} | macroF1 {row['macroF1']:.3f}")

stress = pd.DataFrame(stress_rows).set_index("model")[["det_Se", "det_+P"] + metric_cols]
display(stress.round(3))
stress.round(4).to_csv("/kaggle/working/export/stress_tests.csv")

In [ ]:
DEPLOY_INCLUDE_INCART = True

def concat(*Ds):
    out = {k: np.concatenate([d[k] for d in Ds]) for k in ["W", "RR", "y", "rec", "T"]}
    out.update(Wt=torch.tensor(out["W"], device=DEV), RRt=torch.tensor(out["RR"], device=DEV),
               yt=torch.tensor(out["y"], device=DEV), Tt=torch.tensor(out["T"], device=DEV))
    return out

DEP = concat(*([mit] + ([incart] if DEPLOY_INCLUDE_INCART else [])))
dep_idx = np.arange(len(DEP["y"]))
print("deployment training beats:", len(dep_idx), counts(DEP))

deploy_models = []
for s in tqdm(SEEDS, desc="deployment models"):
    seed_all(s)
    deploy_models.append(train_fixed(CANDIDATES[MODEL_NAME], DEP, dep_idx, CFG_FINAL,
                                     desc=f"deploy seed {s}").eval())

In [ ]:
import copy, subprocess, sys, shutil

class BeatEnsemble(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x, rr):
        return torch.stack([torch.softmax(m(x, rr), 1) for m in self.models]).mean(0)

EXPORT = "/kaggle/working/export"
ens = BeatEnsemble([copy.deepcopy(m).cpu().eval() for m in deploy_models]).eval()
torch.save(ens.state_dict(), f"{EXPORT}/beat_model.pth")

ex_x, ex_rr = torch.randn(8, 2, L), torch.randn(8, 7)
with torch.no_grad():
    torch.jit.trace(ens, (ex_x, ex_rr)).save(f"{EXPORT}/beat_model_ts.pt")
    tx, trr = torch.randn(37, 2, L), torch.randn(37, 7)
    ts_diff = (torch.jit.load(f"{EXPORT}/beat_model_ts.pt")(tx, trr) - ens(tx, trr)).abs().max().item()
print("TorchScript vs eager max diff:", ts_diff)

try:
    try:
        import onnx, onnxruntime
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
        import onnx, onnxruntime
    kw = dict(input_names=["beat", "rr"], output_names=["probs"], opset_version=17,
              dynamic_axes={"beat": {0: "batch"}, "rr": {0: "batch"}, "probs": {0: "batch"}})
    try:
        torch.onnx.export(ens, (ex_x, ex_rr), f"{EXPORT}/beat_model.onnx", dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(ens, (ex_x, ex_rr), f"{EXPORT}/beat_model.onnx", **kw)
    sess = onnxruntime.InferenceSession(f"{EXPORT}/beat_model.onnx")
    onnx_out = sess.run(None, {"beat": tx.numpy(), "rr": trr.numpy()})[0]
    print("ONNX vs eager max diff:", np.abs(onnx_out - ens(tx, trr).detach().numpy()).max())
except Exception as e:
    print("ONNX export skipped:", e)

In [ ]:
P_ds2, y_ds2 = np.mean(probs["DS2"], 0), mit["y"][ds2_idx]
rows = []
for t in tqdm([0.4, 0.5, 0.6, 0.7, 0.8], desc="thresholds"):
    keep = P_ds2.max(1) >= t
    rows.append(dict(threshold=t, coverage=keep.mean(),
                     acc_kept=(P_ds2.argmax(1)[keep] == y_ds2[keep]).mean(),
                     acc_flagged_unknown=(P_ds2.argmax(1)[~keep] == y_ds2[~keep]).mean() if (~keep).any() else np.nan))
thr_df = pd.DataFrame(rows)
display(thr_df.round(3))
UNKNOWN_T = 0.5

In [ ]:
from pathlib import Path

MODULE_SRC = r'''"""CardioSense AI beat classifier.

Shared by the training notebook and the FastAPI backend so preprocessing is
identical in both places. Requires numpy, scipy and torch.
"""
import json
from math import gcd

import numpy as np
from scipy.signal import butter, find_peaks, resample_poly, sosfiltfilt

CLASSES = ["N", "S", "V", "F"]
CLASS_NAMES = {"N": "Normal", "S": "Supraventricular ectopic",
               "V": "Ventricular ectopic", "F": "Fusion"}


def preprocess(x, fs_in, fs_out=250, band=(0.5, 40.0), order=3):
    x = np.asarray(x, dtype=np.float64)
    if fs_in != fs_out:
        g = gcd(int(fs_in), int(fs_out))
        x = resample_poly(x, int(fs_out) // g, int(fs_in) // g)
    sos = butter(order, list(band), btype="band", fs=fs_out, output="sos")
    return sosfiltfilt(sos, x)


def causal_mean(v, n):
    c = np.cumsum(np.r_[0.0, v])
    i = np.arange(len(v))
    s = np.maximum(0, i - n + 1)
    return (c[i + 1] - c[s]) / (i + 1 - s)


def rr_features(r, fs=250):
    rr = np.diff(r) / fs
    pre = np.r_[rr[0], rr]
    post = np.r_[rr, rr[-1]]
    loc10, loc100 = causal_mean(pre, 10), causal_mean(pre, 100)
    f = np.stack([pre, post, loc10, pre / loc10, post / loc10, pre / post, pre / loc100], 1)
    return np.log(np.clip(f, 1e-2, 10)).astype(np.float32)


def zscore(W):
    return (W - np.median(W, 1, keepdims=True)) / (W.std(1, keepdims=True) + 1e-6)


def running_template(Z, n_prev=20):
    T = np.empty_like(Z)
    T[0] = Z[0]
    for k in range(1, len(Z)):
        T[k] = np.median(Z[max(0, k - n_prev):k], 0)
    return T


def detect_r_peaks(x, fs=250):
    sos = butter(2, [5, 18], btype="band", fs=fs, output="sos")
    y = sosfiltfilt(sos, x)
    e = np.gradient(y) ** 2
    w = max(1, int(round(0.12 * fs)))
    integ = np.convolve(e, np.ones(w) / w, mode="same")
    blk = int(5 * fs)
    thr = np.empty_like(integ)
    for s in range(0, len(integ), blk):
        thr[s:s + blk] = 0.3 * np.percentile(integ[max(0, s - 2 * blk):s + 3 * blk], 98)
    cand, _ = find_peaks(integ, distance=int(0.25 * fs))
    cand = cand[integ[cand] > thr[cand]]
    half = int(0.08 * fs)
    r = [max(0, p - half) + int(np.argmax(np.abs(x[max(0, p - half):p + half + 1]))) for p in cand]
    keep = []
    for i in sorted(set(r)):
        if not keep or i - keep[-1] > int(0.2 * fs):
            keep.append(i)
        elif abs(x[i]) > abs(x[keep[-1]]):
            keep[-1] = i
    return np.array(keep, dtype=int)


def _model_norm(a):
    # matches torch.median (lower middle) and torch.std (ddof=1) used in training
    k = (a.shape[1] - 1) // 2
    med = np.partition(a, k, axis=1)[:, k:k + 1]
    return (a - med) / (a.std(1, ddof=1, keepdims=True) + 1e-6)


def build_inputs(x, r, cfg):
    pre, post, margin, fs = cfg["pre"], cfg["post"], cfg["margin"], cfg["fs"]
    L = pre + post
    rr = rr_features(r, fs)
    ok = (r - pre - margin >= 0) & (r + post + margin <= len(x))
    if not ok.any():
        return None, None, ok
    W = x[r[ok, None] + np.arange(-pre - margin, post + margin)[None, :]].astype(np.float32)
    T = running_template(zscore(W), cfg["template_prev_beats"])
    xc = _model_norm(W[:, margin:margin + L])
    tc = _model_norm(T[:, margin:margin + L])
    return np.stack([xc, xc - tc], 1).astype(np.float32), rr[ok], ok


class BeatClassifier:
    def __init__(self, model_path, config_path, device="cpu"):
        import torch
        self._torch = torch
        with open(config_path) as f:
            self.cfg = json.load(f)
        self.device = device
        self.model = torch.jit.load(model_path, map_location=device).eval()

    def predict_inputs(self, beats, rr, batch=2048):
        torch = self._torch
        out = []
        with torch.no_grad():
            for i in range(0, len(beats), batch):
                b = torch.from_numpy(beats[i:i + batch]).to(self.device)
                f = torch.from_numpy(rr[i:i + batch]).to(self.device)
                out.append(self.model(b, f).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, len(CLASSES)), np.float32)

    def classify(self, raw, fs_in):
        cfg = self.cfg
        x = preprocess(raw, fs_in, cfg["fs"], tuple(cfg["bandpass_hz"]), cfg["filter_order"])
        r = detect_r_peaks(x, cfg["fs"])
        if len(r) < 2:
            return []
        beats, rr, ok = build_inputs(x, r, cfg)
        if beats is None:
            return []
        P = self.predict_inputs(beats, rr)
        idx = (np.log(P + 1e-9) + np.asarray(cfg["class_bias"])).argmax(1)
        results = []
        for pos, p, k in zip(r[ok], P, idx):
            conf = float(p[k])
            results.append({
                "sample": int(pos),
                "time_s": float(pos / cfg["fs"]),
                "label": CLASSES[k] if conf >= cfg["unknown_threshold"] else "Unknown",
                "confidence": conf,
                "probs": {c: float(v) for c, v in zip(CLASSES, p)},
                "model_version": cfg["model_version"],
            })
        return results
'''
Path(f"{EXPORT}/cardiosense_beats.py").write_text(MODULE_SRC)

def flat(d):
    return {k: float(v) if isinstance(v, (np.floating, float, int, np.integer)) else v for k, v in d.items()}

config = {
    "model_version": f"cardiosense-beat-resnet-tpl-{time.strftime('%Y%m%d')}",
    "architecture": "ResNet1D, 2-channel input (beat, beat minus patient template) + 7 RR features; "
                    "3-seed ensemble, SWA weights, softmax-averaged",
    "files": {"torchscript": "beat_model_ts.pt", "state_dict": "beat_model.pth", "onnx": "beat_model.onnx"},
    "classes": CLASSES, "fs": FS, "pre": PRE, "post": POST, "margin": MARGIN, "window_len": L,
    "bandpass_hz": [0.5, 40.0], "filter_order": 3, "template_prev_beats": 20,
    "rr_features": ["pre_rr", "post_rr", "local10_rr", "pre/local10", "post/local10", "pre/post", "pre/local100"],
    "rr_transform": "log(clip(x, 0.01, 10)), RR in seconds, causal local means",
    "inputs": {"beat": "(B, 2, 250) float32", "rr": "(B, 7) float32"}, "output": "(B, 4) class probabilities",
    "unknown_threshold": UNKNOWN_T, "class_bias": FINAL["class_bias"],
    "latency_note": "post_rr needs the next beat, so each label is available ~1 beat after the R-peak",
    "training": {"deployment_data": "MIT-BIH 44 non-paced records" + (" + INCART 75 records" if DEPLOY_INCLUDE_INCART else ""),
                 "augmentation": "R-peak jitter +-32 ms, NSTDB em/ma/bw noise at 6-30 dB SNR",
                 "cfg": CFG_FINAL, "seeds": SEEDS},
    "evaluation_note": "Reported metrics come from models trained on DS1 only (patient-wise); "
                       "the deployment ensemble is retrained on all data and has no held-out test of its own.",
    "evaluation": {
        "cv_ds1_5fold": flat(summary3.loc[best_row].to_dict()),
        "ds2_mean_3_seeds": {f"{a}_{b}": float(v) for (a, b), v in agg.loc["DS2"].items()},
        "incart_mean_3_seeds": {f"{a}_{b}": float(v) for (a, b), v in agg.loc["INCART"].items()},
        "beatwise_70_30": flat({k: v for k, v in row_b.items() if k != "model"}),
        "stress_tests_ds2": stress.reset_index().to_dict(orient="records"),
        "unknown_threshold_table_ds2": thr_df.to_dict(orient="records"),
    },
    "limitations": [
        "Single-lead (MLII / lead II) only; not for ischaemia or 12-lead interpretation.",
        "Supraventricular detection is weak when ectopic beats dominate the rhythm (MIT-BIH record 232); "
        "rhythm-level rules (pauses, bradycardia) must cover this case.",
        "Fusion (F) beats are rarely detected on unseen patients.",
        "Not validated on AD8232 recordings yet; research and demonstration use only, not a medical device.",
    ],
}
json.dump(config, open(f"{EXPORT}/model_config.json", "w"), indent=2, default=float)
print("written:", sorted(os.listdir(EXPORT)))

In [ ]:
import importlib
sys.path.insert(0, EXPORT)
import cardiosense_beats as cb
importlib.reload(cb)

clf = cb.BeatClassifier(f"{EXPORT}/beat_model_ts.pt", f"{EXPORT}/model_config.json")
for rec in tqdm(["100", "119", "233"], desc="backend check"):
    s, a = wfdb.rdrecord(f"{DATA}/mitdb/{rec}"), wfdb.rdann(f"{DATA}/mitdb/{rec}", "atr")
    t0 = time.time()
    res = clf.classify(s.p_signal[:, s.sig_name.index("MLII")], s.fs)
    dt = time.time() - t0
    sym = np.asarray(a.symbol)
    beat = np.isin(sym, BEAT_SYMBOLS)
    ref = np.round(a.sample[beat] * FS / s.fs).astype(int)
    ref_lab = np.array([AAMI[c] for c in sym[beat]])
    det = np.array([b["sample"] for b in res])
    pairs = match_peaks(det, ref, int(0.15 * FS))
    agree = np.mean([res[i]["label"] == ref_lab[j] for i, j in pairs if ref_lab[j] in CLASSES])
    unknown = np.mean([b["label"] == "Unknown" for b in res])
    tqdm.write(f"record {rec}: {len(res)} beats in {dt:.1f}s (30-min recording, CPU) | "
               f"label agreement {agree:.3f} | unknown {unknown:.3f}")

print(json.dumps(res[100], indent=2))
shutil.make_archive("/kaggle/working/cardiosense_beat_model", "zip", EXPORT)
for f in sorted(os.listdir(EXPORT)):
    print(f"{os.path.getsize(f'{EXPORT}/{f}') / 1e6:8.2f} MB  {f}")